# Session 2: from a balanced equation to grams
**A. Real-world problem.** A balanced equation counts molecules, but a shop sells kilograms.
To neutralise a spill of sulphuric acid with limestone you need to know how many kilograms
of limestone, not how many molecules. The bridge between molecules and grams is the mole.

**Plan.** (1) molar mass from a table of atomic masses, (2) n = m / M and Avogadro's number,
(3) a general recipe calculator, (4) limiting reagent, theoretical and percent yield,
(5) molarity, (6) the limestone design problem.

In [ ]:
import pathlib
import re

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# standard atomic masses in g/mol (rounded textbook values)
ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "Na": 22.990,
               "Cl": 35.45, "S": 32.06, "Ca": 40.078, "Fe": 55.845, "Al": 26.982}
AVOGADRO = 6.022e23          # entities per mole (6.02214076e23 exactly; 6.022e23 is plenty here)

## Step 1: molar mass
The molar mass M of a substance is the mass of one mole of it, in g/mol. Add the atomic
masses of all atoms in the formula: M(H2O) = 2 * 1.008 + 15.999 = 18.015 g/mol.

In [ ]:
def parse_formula(formula):
    """{element: count} for formulas such as 'CaCO3' or 'Ca(OH)2'."""
    tokens = re.findall(r"[A-Z][a-z]?|\d+|\(|\)", formula)
    if not formula or "".join(tokens) != formula:
        raise ValueError(f"cannot read formula {formula!r}")
    stack, i = [{}], 0
    while i < len(tokens):
        t = tokens[i]
        if t == "(":
            stack.append({})
        elif t == ")":
            if len(stack) == 1:
                raise ValueError(f"unbalanced bracket in {formula!r}")
            group = stack.pop()
            mult = 1
            if i + 1 < len(tokens) and tokens[i + 1].isdigit():
                mult, i = int(tokens[i + 1]), i + 1
            for el, n in group.items():
                stack[-1][el] = stack[-1].get(el, 0) + n * mult
        elif t.isdigit():
            raise ValueError(f"misplaced number in {formula!r}")
        else:
            n = 1
            if i + 1 < len(tokens) and tokens[i + 1].isdigit():
                n, i = int(tokens[i + 1]), i + 1
            stack[-1][t] = stack[-1].get(t, 0) + n
        i += 1
    if len(stack) != 1:
        raise ValueError(f"unbalanced bracket in {formula!r}")
    return stack[0]


def molar_mass(formula, table=ATOMIC_MASS):
    """Molar mass in g/mol."""
    return sum(table[el] * n for el, n in parse_formula(formula).items())

## Step 2: moles, grams and molecules
n = m / M (moles = grams divided by g/mol), and the number of molecules is N = n * N_A,
where N_A = 6.022e23 per mole is Avogadro's number.

In [ ]:
def moles(mass_g, formula):
    return mass_g / molar_mass(formula)


def grams(n_mol, formula):
    return n_mol * molar_mass(formula)


def molecules(n_mol):
    return n_mol * AVOGADRO

## Step 3: the recipe calculator
In a balanced equation the coefficients are mole ratios. If a reaction makes `c_out` moles
of one species for every `c_in` moles of another, then n_out = n_in * c_out / c_in.

In [ ]:
def convert_mass(mass_g, formula_in, coef_in, formula_out, coef_out):
    """Mass (g) of species `out` that matches `mass_g` of species `in` in the balanced equation."""
    n_in = moles(mass_g, formula_in)
    return grams(n_in * coef_out / coef_in, formula_out)

## Step 4: limiting reagent and yield
Reagents run out at different times. For each reactant divide its moles by its coefficient;
the smallest value is the reaction extent (how many "recipes" can be made) and that
reactant is the limiting reagent. Theoretical yield = extent * coefficient * M of the
product. Percent yield = 100 * actual / theoretical.

In [ ]:
def limiting_reagent(reactants, masses_g):
    """reactants = [(coef, formula), ...], masses in g.  Returns (index of limiting reagent, extent in mol)."""
    ratios = [moles(m, f) / c for (c, f), m in zip(reactants, masses_g)]
    k = int(np.argmin(ratios))
    return k, float(ratios[k])


def theoretical_yield(product, extent_mol):
    """product = (coef, formula); returns grams made if the reaction runs to completion."""
    coef, formula = product
    return grams(extent_mol * coef, formula)


def percent_yield(actual_g, theoretical_g):
    return 100.0 * actual_g / theoretical_g

## Step 5: concentration of a solution
Molarity c = n / V in mol/L (written M, not to be confused with molar mass). To make V litres
of a solution of concentration c you weigh m = c * V * M grams of solute.

In [ ]:
def molarity(n_mol, volume_l):
    return n_mol / volume_l


def solute_mass(conc_mol_per_l, volume_l, formula):
    return conc_mol_per_l * volume_l * molar_mass(formula)

## Step 6: design problem, limestone for an acid spill
CaCO3 + H2SO4 -> CaSO4 + H2O + CO2 (balanced as written: 1 : 1). Limestone is not pure
and we add a safety excess so that the spill is surely neutralised.

In [ ]:
def limestone_needed(acid_kg, purity=1.0, excess=0.0):
    """kg of limestone (purity = mass fraction CaCO3, excess = extra fraction) to neutralise acid_kg of H2SO4."""
    pure = convert_mass(acid_kg * 1000.0, "H2SO4", 1, "CaCO3", 1) / 1000.0
    return pure * (1.0 + excess) / purity


def co2_released(limestone_kg, purity=1.0):
    """kg of CO2 released if all the CaCO3 reacts."""
    return convert_mass(limestone_kg * purity * 1000.0, "CaCO3", 1, "CO2", 1) / 1000.0

## Step 7: see it
Left: CO2 from 16 g of CH4 as the oxygen supply grows (a kink where oxygen stops being the
limit). Right: kilograms of limestone against acid mass for three purities.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    meth = [(1, "CH4"), (2, "O2")]
    print("M(CH4) = %.3f g/mol, M(O2) = %.3f, M(CO2) = %.3f, M(H2O) = %.3f" % tuple(
        molar_mass(f) for f in ("CH4", "O2", "CO2", "H2O")))
    n = moles(16.0, "CH4")
    print("16 g CH4 = %.4f mol = %.3e molecules" % (n, molecules(n)))
    co2 = convert_mass(16.0, "CH4", 1, "CO2", 1)
    print("CO2 from 16 g CH4: %.2f g (O2 needed: %.2f g)" % (co2, convert_mass(16.0, "CH4", 1, "O2", 2)))

    o2 = np.linspace(0, 100, 201)
    y = [theoretical_yield((1, "CO2"), limiting_reagent(meth, [16.0, m])[1]) if m > 0 else 0.0 for m in o2]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(o2, y)
    need = convert_mass(16.0, "CH4", 1, "O2", 2)
    ax.axvline(need, color="gray", ls=":")
    ax.annotate("oxygen no longer limits\n(%.1f g O2)" % need, (need, co2 * 0.55), fontsize=8, ha="right")
    ax.set_xlabel("oxygen supplied (g)")
    ax.set_ylabel("CO2 formed from 16 g CH4 (g)")
    ax.set_title("Yield rises until the other reagent becomes limiting")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "yield_vs_oxygen.png", dpi=120)

    acid = np.linspace(0, 1000, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for pur in (1.0, 0.92, 0.80):
        ax2.plot(acid, [limestone_needed(a, pur, 0.10) for a in acid], label="purity %d %%, 10 %% excess" % round(pur * 100))
    ax2.set_xlabel("sulphuric acid to neutralise (kg)")
    ax2.set_ylabel("limestone to order (kg)")
    ax2.set_title("Design: limestone for an acid spill")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "limestone_design.png", dpi=120)
    print("500 kg H2SO4: pure CaCO3 %.1f kg; 92%% pure + 10%% excess: %.1f kg; CO2 %.1f kg" % (
        limestone_needed(500), limestone_needed(500, 0.92, 0.10), co2_released(limestone_needed(500))))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Compute the molar mass of Ca(OH)2 with `molar_mass`. Predict it by hand first (40.078 + 2 * 15.999 + 2 * 1.008).
2. **Stretch.** Use `limiting_reagent` and `theoretical_yield` for 10 g of CH4 and 32 g of O2. Which reagent limits, and how many grams of CO2 and H2O form?
3. **Explain.** Explain why a bag of 90 percent pure limestone must be heavier than the pure CaCO3 the acid needs, and by what factor.

## Assessment
- **A. Conceptual.** In CH4 + 2 O2 -> CO2 + 2 H2O the coefficients are 1 : 2 : 1 : 2. Does 1 g of CH4 need 2 g of O2? Explain.
- **B. Numerical.** How many grams of NaCl do you need for 0.5 L of a 0.10 mol/L solution? How many NaCl formula units is that?
- **C. Graph and simulation interpretation.** In the yield graph (16 g CH4), what is the slope of the rising part and where does the kink occur? Why is the graph flat afterwards?
- **D. Design and optimisation challenge.** Design: 250 kg of sulphuric acid must be neutralised with limestone that is 90 percent CaCO3, with a 15 percent safety excess, in 25 kg bags. How many bags? How much CO2 is released at least? State the balanced equation.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("M(H2O) = 18.015 g/mol (hand: 2 * 1.008 + 15.999)", abs(L.molar_mass("H2O") - 18.015) < 1e-9))
    checks.append(("M(CaCO3) = 100.086 g/mol (hand: 40.078 + 12.011 + 3 * 15.999)", abs(L.molar_mass("CaCO3") - 100.086) < 1e-9))
    checks.append(("M(CH4) = 16.043 and M(CO2) = 44.009", abs(L.molar_mass("CH4") - 16.043) < 1e-9 and abs(L.molar_mass("CO2") - 44.009) < 1e-9))
    checks.append(("18.015 g of water is one mole = 6.022e23 molecules",
                   abs(L.moles(18.015, "H2O") - 1.0) < 1e-12 and abs(L.molecules(1.0) - 6.022e23) < 1e17))
    co2 = L.convert_mass(16.0, "CH4", 1, "CO2", 1)
    checks.append(("worked example: 16 g of CH4 gives 43.9 g of CO2 (43.89 g)", abs(co2 - 43.89) < 0.01))
    h2o = L.convert_mass(16.0, "CH4", 1, "H2O", 2)
    o2 = L.convert_mass(16.0, "CH4", 1, "O2", 2)
    checks.append(("mass is conserved: CH4 + O2 in = CO2 + H2O out", abs(16.0 + o2 - co2 - h2o) < 1e-9))
    meth = [(1, "CH4"), (2, "O2")]
    k, ext = L.limiting_reagent(meth, [40.0, 50.0])
    checks.append(("40 g CH4 with 50 g O2: oxygen limits (index 1), extent 0.7813 mol", k == 1 and abs(ext - 0.7813) < 1e-3))
    k, ext = L.limiting_reagent(meth, [16.0, 200.0])
    checks.append(("16 g CH4 with excess O2: methane limits (index 0)", k == 0))
    k, ext = L.limiting_reagent(meth, [L.grams(1.0, "CH4"), L.grams(2.0, "O2")])
    checks.append(("exact 1 : 2 mole feed is balanced; the extent is 1 mol (edge case)", abs(ext - 1.0) < 1e-12))
    checks.append(("percent yield: 40.0 g actual out of 43.89 g is 91.1 %", abs(L.percent_yield(40.0, co2) - 91.14) < 0.01))
    checks.append(("0.10 M NaCl, 0.5 L needs 2.922 g (hand: 0.05 mol * 58.44)", abs(L.solute_mass(0.10, 0.5, "NaCl") - 2.922) < 1e-3))
    checks.append(("molarity: 0.05 mol in 0.5 L is 0.10 mol/L", abs(L.molarity(0.05, 0.5) - 0.10) < 1e-12))
    checks.append(("limestone for 500 kg acid, pure: 510.3 kg", abs(L.limestone_needed(500.0) - 510.3) < 0.1))
    checks.append(("92 % pure with 10 % excess: 610.1 kg", abs(L.limestone_needed(500.0, 0.92, 0.10) - 610.1) < 0.1))
    checks.append(("limestone scales linearly with acid", abs(L.limestone_needed(1000.0) - 2 * L.limestone_needed(500.0)) < 1e-9))
    acid = 98.072
    checks.append(("one mole of acid (98.072 g) needs one mole of CaCO3 (100.086 g)",
                   abs(L.convert_mass(acid, "H2SO4", 1, "CaCO3", 1) - 100.086) < 1e-6))
    try:
        L.molar_mass("Xx2")
        ok = False
    except KeyError:
        ok = True
    checks.append(("an element missing from the table raises KeyError", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
